# Week 5 (starter): RAG Pipeline with Retrieval Evaluation

Retrieval runs fully local and free; only generation needs a key (set `GEMINI_API_KEY`, else it is skipped with a notice). Cells marked **TODO (you)** are yours. Dependencies: `sentence-transformers`, `faiss-cpu`. For generation: `pip install openai`.

In [1]:
import os, re, pathlib, numpy as np
os.environ['HF_HOME'] = str((pathlib.Path('.') / '.hf_cache').resolve())
from sentence_transformers import SentenceTransformer
import faiss
embedder = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

c:\Users\wyatt\Documents\Gradschool\cosc_650\repos\cosc-650-applied-llm-systems\src\week5\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7021.98it/s]


## Part 1: Build the pipeline
Ingest and chunk a documentation corpus of at least eight documents, embed the chunks locally with sentence-transformers, and store them in a vector index (FAISS or similar). Implement retrieval that embeds a query and returns the top-k chunks.

In [2]:
from pypdf import PdfReader

DOCS_DIR = pathlib.Path("gardening_docs")

def load_pdf(file_path):
    reader = PdfReader(file_path)
    pages = []

    for page in reader.pages:
        text = page.extract_text()

        if text:
            pages.append(text)

    return "\n\n".join(pages)


documents = []

for file_path in DOCS_DIR.glob("*.pdf"):
    documents.append({
        "source": file_path.name,
        "text": load_pdf(file_path)
    })


print(f"Loaded {len(documents)} documents")

for doc in documents:
    print(f"{doc['source']}: {len(doc['text']):,} characters")

Loaded 8 documents
bush_gardening.pdf: 147,544 characters
fruit_gardening.pdf: 24,727 characters
gardening_beds.pdf: 20,128 characters
gardening_pests.pdf: 10,832 characters
gardening_planters.pdf: 34,943 characters
grass_gardening.pdf: 33,280 characters
tree_planting.pdf: 23,515 characters
vegetable_gardening.pdf: 78,865 characters


In [3]:
def chunk_fixed(text, size, overlap):
    text = re.sub(r'\s+', ' ', text).strip(); out, i = [], 0
    while i < len(text):
        out.append(text[i:i+size]); i += size - overlap
    return out

def chunk_paragraph(text):
    return [re.sub(r'\s+',' ',p).strip() for p in text.split('\n\n') if p.strip()]


def chunk_documents(documents, chunker):
    chunks = []

    for doc in documents:
        doc_chunks = chunker(doc["text"])

        for chunk_id, chunk in enumerate(doc_chunks):
            chunks.append({
                "source": doc["source"],
                "chunk_id": chunk_id,
                "text": chunk
            })

    return chunks


configs = {
    "small (120/20)": chunk_documents(
        documents,
        lambda text: chunk_fixed(text, 120, 20)
    ),
    "large (320/40)": chunk_documents(
        documents,
        lambda text: chunk_fixed(text, 320, 40)
    ),
    "by-paragraph": chunk_documents(
        documents,
        chunk_paragraph
    )
}

for n, c in configs.items():
    print(f'{n:16s} -> {len(c)} chunks')

def build(chunks):
    texts = [chunk["text"] for chunk in chunks]

    e = embedder.encode(
        texts,
        normalize_embeddings=True
    ).astype("float32")

    idx = faiss.IndexFlatIP(e.shape[1])
    idx.add(e)

    return idx

# def retrieve(idx, chunks, q, k=3):
#     qe = embedder.encode([q], normalize_embeddings=True).astype('float32')
#     # FAISS uses -1 for missing results when fewer than k chunks exist.
#     return [chunks[i] for i in idx.search(qe, k)[1][0] if i >= 0]

def retrieve(idx, chunks, q, k=3):
    qe = embedder.encode(
        [q],
        normalize_embeddings=True
    ).astype("float32")

    scores, indices = idx.search(qe, k)

    results = []

    for score, i in zip(scores[0], indices[0]):
        if i >= 0:
            results.append({
                "source": chunks[i]["source"],
                "chunk_id": chunks[i]["chunk_id"],
                "text": chunks[i]["text"],
                "score": float(score)
            })

    return results

small (120/20)   -> 3668 chunks
large (320/40)   -> 1313 chunks
by-paragraph     -> 200 chunks


In [4]:
indices = {
    name: build(chunks)
    for name, chunks in configs.items()
}

for name, idx in indices.items():
    print(f"{name:16s} -> {idx.ntotal} vectors")

small (120/20)   -> 3668 vectors
large (320/40)   -> 1313 vectors
by-paragraph     -> 200 vectors


In [5]:
query = "How much sunlight do vegetables need?"

results = retrieve(
    indices["large (320/40)"],
    configs["large (320/40)"],
    query,
    k=3
)

for i, result in enumerate(results, start=1):
    print(f"Result {i}")
    print(f"Source: {result['source']}")
    print(f"Chunk: {result['chunk_id']}")
    print(f"Score: {result['score']:.4f}")
    print(f"Text: {result['text']}")
    print()

Result 1
Source: vegetable_gardening.pdf
Chunk: 10
Score: 0.7211
Text: ood tilth. Sunlight. Most summer vegetable crops need at least eight hours of direct sunlight a day. If trees or shrubs are shading your garden, Vegetable Gardening 6-3 prune branches to allow in more light. If a solid fence is the source of the shade, consider replacing it with a post and rail fence that will let in m

Result 2
Source: bush_gardening.pdf
Chunk: 203
Score: 0.5769
Text: ount of available sunlight will affect species selection for a particular location. Most woody plants require full sunlight (at least 6 hours per day) for optimal growth and flower bloom. Some do well in light shade, and a few shrub species perform well in dense shade. Exposure to wind is also a consideration. Wind can

Result 3
Source: vegetable_gardening.pdf
Chunk: 12
Score: 0.5425
Text: rops with low light requirements, such as leafy green crops like lettuce, or asparagus, garlic, and leeks. If these do well, expand into medium lig

## Part 2: Evaluate retrieval
Compare at least two embedding models, or at least three chunking configurations, on at least ten test queries for which you decide in advance which retrieved chunks count as relevant. Report precision and recall at the chunk level, and explain where the choices disagree and why.

In [ ]:
# TODO (you): your own queries and the fact each answer must contain.
test_queries = [
    "How much sunlight do vegetables need?",
    "How should I choose a location for a vegetable garden?",
    "What type of soil is best for planting fruit trees?",
    "How should fruit trees be watered after planting?",
    "What are the benefits of using raised garden beds?",
    "How wide should a raised garden bed be?",
    "How can crop rotation help control garden insects?",
    "How should shrubs be selected for different sunlight conditions?",
    "What should I consider when selecting a tree planting site?",
    "How should a lawn be watered and maintained?"
]
# This example checks whether retrieved context contains the answer.
# It does not calculate the chunk-level precision and recall required below.
for n, c in configs.items():
    hits = sum(1 for q, fact in qa if fact.lower() in ' '.join(retrieve(indices[n], c, q)).lower())
    print(f'{n:16s} answer-recall@3: {hits}/{len(qa)} = {hits/len(qa):.2f}')
# TODO (you): compare at least two embedding models OR at least three chunking
# configurations on at least ten queries. Label relevant chunks in advance,
# report chunk-level precision and recall, and explain where the choices disagree.

## Part 3: Generate (needs a key)
Build a grounded prompt from the retrieved chunks and answer with Gemini.

In [ ]:
def gemini_chat(messages, model='gemini-2.5-flash', **kw):
    if not os.environ.get('GEMINI_API_KEY'):
        return None
    from openai import OpenAI
    client = OpenAI(api_key=os.environ['GEMINI_API_KEY'], base_url='https://generativelanguage.googleapis.com/v1beta/openai/')
    return client.chat.completions.create(model=model, messages=messages, **kw).choices[0].message.content

def answer(q, config='by-paragraph'):
    ctx = retrieve(indices[config], configs[config], q, k=3)
    prompt = 'Answer only from the context.\nContext:\n- ' + '\n- '.join(ctx) + f'\nQuestion: {q}\nAnswer:'
    out = gemini_chat([{'role':'user','content':prompt}])
    return out if out is not None else '[API-BLOCKED] set GEMINI_API_KEY to generate; grounded prompt was built'

print(answer('what port does orbit use'))

## Part 4 and 5: failure, submit
**TODO (you):** show a query where retrieval surfaces the wrong chunk or a chunk config splits a fact, and explain the mitigation. Then open a pull request with a result summary and review a classmate's Week 5 PR (the term's formal peer review). Use the rubric attached to the Canvas assignment for grading criteria and point values.